# Probability Foundations Through Simulation

**Official MA1001B Alignment:** *1.1 set theory and probability calculation; 1.2 counting techniques.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Represent compound random events as Boolean conditions (`and`, `or`, `not`) in Pandas.
- Estimate empirical probabilities through large-scale Monte Carlo simulation.
- Calculate exact theoretical probabilities using combinatorial sample spaces and multi-indexing.
- Evaluate the long-run convergence of empirical simulation estimates to theoretical probabilities.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model random outcomes under uncertainty to understand long-run event frequencies and probability rules.
- **2. Computational Link (How Python represents it):** We use vectorized Boolean logic in Pandas to define events and compute empirical proportions across simulated trials.
- **3. Decision Link (How it guides action):** Accurate probability estimates allow the game designer to establish financially viable and fair payout rules.


## Decision Scenario

> **The Problem:** A game designer wants to set a fair payout for an event involving two dice. Before choosing a payout, the designer needs to know how often the event happens and how simulation compares with exact counting.


## Conceptual Explanation

Probability describes long-run regularity under a model. In data science, we often estimate probability from data, but we also use probability models to reason before data are collected. Events can be represented as Boolean conditions. Unions, intersections, and complements become `or`, `and`, and `not` operations in code.


## Mathematical Anchor

For equally likely outcomes, P(A) = number of outcomes in A / number of possible outcomes. For two events, P(A union B) = P(A) + P(B) - P(A intersection B).


## Data And Workflow Notes

This lesson uses simulated dice because the true sample space is known. That makes it possible to compare empirical and theoretical probability.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Simulation Setup: Rolling Two Dice

We simulate 20,000 independent rolls of two fair six-sided dice using NumPy's random integer generator and store the outcomes in a DataFrame.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Simulate 20,000 rolls of two fair six-sided dice (integers from 1 to 6 inclusive)
rolls = pd.DataFrame({
    "die_1": rng.integers(1, 7, size=20_000),
    "die_2": rng.integers(1, 7, size=20_000),
})
# Calculate total sum of the two dice
rolls["total"] = rolls["die_1"] + rolls["die_2"]
rolls.head()


### Step 2: Empirical Probability Estimation via Boolean Logic

We define specific events (rolling a total of 7, rolling doubles) using Boolean masks and estimate their probabilities by calculating the mean of the boolean series.


In [ ]:
# Define events as Boolean Series (True/False)
event_total_7 = rolls["total"].eq(7)
event_double = rolls["die_1"].eq(rolls["die_2"])

# Estimate empirical probabilities (mean of boolean series equals proportion of True values)
probability_summary = pd.Series({
    "P(total = 7)": event_total_7.mean(),
    "P(double)": event_double.mean(),
    "P(total = 7 and double)": (event_total_7 & event_double).mean(),
    "P(total = 7 or double)": (event_total_7 | event_double).mean(),
})
probability_summary.round(4)


### Step 3: Exact Theoretical Calculation via Sample Space

We construct the exact combinatorial sample space of all 36 possible outcomes using `pd.MultiIndex` to calculate the exact theoretical probabilities.


In [ ]:
# Generate all 36 possible dice combinations (6 x 6 grid)
exact_outcomes = pd.MultiIndex.from_product(
    [range(1, 7), range(1, 7)],
    names=["die_1", "die_2"],
).to_frame(index=False)
exact_outcomes["total"] = exact_outcomes["die_1"] + exact_outcomes["die_2"]

# Define exact Boolean conditions on the complete sample space
exact_total_7 = exact_outcomes["total"].eq(7)
exact_double = exact_outcomes["die_1"].eq(exact_outcomes["die_2"])

# Calculate exact theoretical probabilities
pd.Series({
    "Exact P(total = 7)": exact_total_7.mean(),
    "Exact P(double)": exact_double.mean(),
    "Exact P(total = 7 or double)": (exact_total_7 | exact_double).mean(),
}).round(4)


### Step 4: Analyzing Long-Run Convergence

We demonstrate the Law of Large Numbers by tracking how the empirical probability estimate of rolling a 7 converges to the exact probability (0.1667) as sample size increases.


In [ ]:
# Track probability estimation across increasing sample sizes
sample_sizes = [50, 100, 500, 1000, 5000, 20_000]
convergence = []
for n in sample_sizes:
    sample = rolls.head(n)
    convergence.append({
        "sample_size_n": n,
        "estimated_P_total_7": sample["total"].eq(7).mean(),
        "exact_probability": 6 / 36,
        "absolute_error": abs(sample["total"].eq(7).mean() - (6 / 36))
    })

pd.DataFrame(convergence).round(4)


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Explain why the simulated estimate changes with sample size but the exact probability does not.

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Confusing an empirical simulation estimate with the exact mathematical probability.
- **Warning:** Adding P(A) and P(B) for overlapping events without subtracting the intersection P(A and B).
- **Warning:** Ignoring the underlying model assumption that the dice are fair and independent.


## Independent Practice

> [!TIP]
> **Your Task:**
> Define a new event, such as `total >= 10`. Estimate it by simulation, compute it exactly across the 36-outcome sample space, and compare the two values.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** When in real-world data science would simulation be more useful or feasible than exact combinatorial counting?

*Write your brief conceptual reflection below:*
